In [ ]:
%run 05_embeddings.ipynb

# Both functions now available
vec = embed("I love action movies")
print(vec.shape)  # should print (384,)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
#from src.embeddings import embed

def preference_strength(genres, mood, favorite):
    score = len(genres) * 0.3
    score += 1.0 if mood else 0
    score += 1.5 if favorite else 0
    return max(score, 0.1)

def compute_weights(user_a, user_b):
    sa = preference_strength(user_a["genres"], user_a["mood"], user_a["favorite"])
    sb = preference_strength(user_b["genres"], user_b["mood"], user_b["favorite"])
    total = sa + sb
    return round(sa / total, 2), round(sb / total, 2)

def fuse_vectors(vec_a, vec_b, alpha, beta):
    fused = alpha * np.array(vec_a) + beta * np.array(vec_b)
    return fused / np.linalg.norm(fused)

def compatibility_score(vec_a, vec_b):
    score = cosine_similarity([vec_a], [vec_b])[0][0]
    pct = round(float(score) * 100)
    if pct >= 75: label = "Movie soulmates"
    elif pct >= 60: label = "Great match"
    elif pct >= 40: label = "Some overlap"
    else: label = "Opposites — challenge mode"
    return pct, label

In [ ]:
# Test 1 — compute_weights: stronger preferences get higher weight
user_a = {"genres": ["Action", "Sci-Fi"], "mood": "Adventurous", "favorite": "Interstellar"}
user_b = {"genres": ["Romance"], "mood": "Romantic", "favorite": None}

alpha, beta = compute_weights(user_a, user_b)
print("Test 1 — Compute weights:")
print(f"Person A weight: {alpha}")
print(f"Person B weight: {beta}")
print(f"Weights sum to 1: {round(alpha + beta, 2) == 1.0}")
print(f"Person A stronger (more genres + favorite): {alpha > beta}")
print()

In [ ]:
# Test 2 — compatibility_score: similar users score higher than opposites
%run 05_embeddings.ipynb

vec_a = embed("I love action sci-fi adventure movies")
vec_b_similar = embed("I enjoy action films and science fiction")
vec_b_opposite = embed("I only watch romantic comedies and love stories")

score_similar, label_similar = compatibility_score(vec_a, vec_b_similar)
score_opposite, label_opposite = compatibility_score(vec_a, vec_b_opposite)

print("Test 2 — Compatibility score:")
print(f"Similar users: {score_similar}% — {label_similar}")
print(f"Opposite users: {score_opposite}% — {label_opposite}")
print(f"Similar scores higher than opposite: {score_similar > score_opposite}")